## General lab instructions
For every lab in this course, you must submit **two items**:

### **1. A Jupyter Notebook (`.ipynb`)**

Your notebook should:

* Contain **all code** needed to run your experiments
* Produce **all figures, tables, and outputs** used in your report
* Be **fully runnable from top to bottom** on a fresh kernel
* Include clear comments explaining your reasoning and workflow

The notebook will be evaluated both for **correctness** and **clarity**.

---

### **2. A Formal Lab Report (PDF or Markdown-to-PDF)**

Your report should:

* Be **self-contained** — a reader should understand everything without opening your notebook
* Present results in a **clean, professional style**
* Include:

  * All asked-for figures with captions
  * All asked-for tables summarizing performance
  * Answers to all discussion questions 


---

### 🔍 Academic Expectations

You must be prepared to:

* **Explain your code**
* **Walk through your debugging steps**
* **Justify your modeling decisions**

Some future quizzes may include questions **directly about the code you were supposed to write**, so make sure you understand every part of your implementation.



# Lab 2: Implementing Logistic Regression and Custom Neural Components

In this lab, you will build key components of modern machine learning models **from scratch**. You will begin by implementing a full multiclass logistic regression model—including the **loss layer**—and training it using both SGD and Adam. You will then extend these ideas to create your own multilayer perceptron (MLP) using custom modules.  

 
---

## What You Will Implement

### 🔹 Part 1 — Multiclass Logistic Regression (Fully Implemented by You)

You will write a complete multiclass logistic regression model, including:

* **Linear forward pass**
  ( z = XW + b )
* **Softmax function**
  ( \text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}} )
* **Cross-entropy loss (forward)**
  ( L = -\log(\text{softmax}(z)_{y}) )
* **Backward pass for the entire loss layer**
  Derive and implement
  [
  \frac{\partial L}{\partial z} = \text{softmax}(z) - \text{one_hot}(y)
  ]
* **Parameter updates using:**

  * **SGD** that you implement
  * **Adam** that you also implement (not PyTorch’s version)

You will train your model on:

* **Linearly separable blobs**
* **Overlapping ellipsoids**
* **Spiral dataset**

You will visualize all decision boundaries and compare performance across datasets and optimizers.

---

## 🔹 Part 2 — Building Your Own MLP

You will extend the ideas from Part 1 to create a multilayer neural network.

You will:

* Use the provided **Linear layer** as a reference
* Implement your own **ReLU activation** (forward + backward)
* Assemble these into an MLP architecture of your choice
* Train the MLP using your own:

  * **SGD optimizer**
  * **Adam optimizer**
* Reuse your **loss layer implementation** from Part 1
  (your MLP should call your loss code, not PyTorch’s)

You must demonstrate:

* Different depths
* Different widths
* Stability of training
* Comparison of SGD vs Adam

You will again train on the three datasets and record results in tables.

 


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import math
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

import numpy as np
import matplotlib.pyplot as plt


# Part 2 — Implementing Your Own Backward Pass (ReLU + Loss) for an MLP

In the previous lab, you replaced PyTorch’s built-in operations with **your own Linear layer** and **your own forward activations**.
Now you will extend this work by implementing **both forward and backward logic** for:

1. **ReLU activation**
2. **Multiclass cross-entropy loss**

This is your first full implementation of **backpropagation inside a neural network**, written entirely with your own functions.

By the end of this part, your entire MLP training loop should use:

* your custom **Linear**
* your custom **ReLU**
* your custom **cross-entropy loss**
* your custom **SGD** 

---

## What You Will Do in This Part

You will:

1. **Implement ReLU forward + backward** using `torch.autograd.Function`
2. **Implement multiclass cross-entropy loss forward + backward** using `torch.autograd.Function`
3. Integrate your custom ReLU and Loss into the MLP you already wrote
4. Integrate SGD into the optimizer class shell provided
5. Train the MLP on all three datasets:

   * blobs
   * ellipsoids
   * spirals 

You should *reuse your existing MLP* and only replace the components listed above.

---

## Required Training Curves (same as Part 1)

For each dataset, you must also produce **two training curves**:

1. **Loss vs iteration**
2. **Misclassification rate vs iteration**

Each must:

* have labeled axes
* have informative titles
* include a short, meaningful caption explaining what the curves show
* be clear and readable

Your training curves should resemble the plots you produced in Part 1 (SGD only for this part).

---

## Deliverables for Part 2

You must submit:

### **1. Your custom ReLU implementation**

Using `torch.autograd.Function`, with:

* `forward(ctx, x)`
* `backward(ctx, grad_output)`

### **2. Your custom multiclass cross-entropy loss**

Also using `torch.autograd.Function`, with:

* `forward(ctx, logits, targets)`
* `backward(ctx, grad_output)`

### **3. Your updated MLP using only your custom components**

### **4. A 1×3 decision-boundary figure**

(one for each dataset)

### **5. A 2×3 grid of training curves**

Three datasets, each showing:

* loss curve
* misclassification curve

You may reuse the helper functions you built in Part 1.
 


## **Discussion Questions**

1. Compare the training curves (loss and misclassification) of the MLP with those of logistic regression. How does the convergence speed differ between the two models across the three datasets?

2. How does dataset difficulty (blobs vs. ellipsoids vs. spirals) affect the training behavior of each model? Do either model show plateaus or oscillations in training, and what might explain these behaviors?

---

If you'd like, I can also provide a short “what good answers might mention” section for instructors.


4. **Compare the loss and misclassification curves across the datasets.
   Which dataset is hardest for the MLP to learn, and how do you know?**


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


# ============================================================
# Simple SGD Optimizer (works like PyTorch)
# ============================================================
class SGD():
    def __init__(self, params, lr=1e-2):
        self.params = list(params)
        self.lr = lr

    def step(self):
        for p in self.params:
            if p.grad is None:
                continue
            p.data -= self.lr * p.grad   # do NOT use p.grad.data

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()


class Adam():
    #fill me in
    pass




# ============================================================
# Custom ReLU Autograd
# ============================================================
class ReLU(nn.Module):
    class ReLUFunction(torch.autograd.Function):

        @staticmethod
        def forward(ctx, x):
            #fill me in

        @staticmethod
        def backward(ctx, grad_out):
            #fill me in

    def forward(self, x):
        return ReLU.ReLUFunction.apply(x)



# ============================================================
# Custom Cross-Entropy Loss
# ============================================================
class CrossEntropyLoss(nn.Module):
    class CEFunction(torch.autograd.Function):

        @staticmethod
        def forward(ctx, logits, y):
            shifted = logits - logits.max(dim=1, keepdims=True).values
            expz = shifted.exp()
            probs = expz / expz.sum(dim=1, keepdims=True)

            ctx.save_for_backward(probs, y)

            N = logits.size(0)
            loss = -torch.log(probs[torch.arange(N), y] + 1e-12).mean()
            return loss

        @staticmethod
        def backward(ctx, grad_out):
            probs, y = ctx.saved_tensors
            N, C = probs.shape

            Y = torch.zeros_like(probs)
            Y[torch.arange(N), y] = 1.0

            grad_logits = (probs - Y) / N
            grad_logits *= grad_out  # scalar

            return grad_logits, None

    def forward(self, logits, y):
        return CrossEntropyLoss.CEFunction.apply(logits, y)



# ============================================================
# Custom Linear Layer (fully correct autograd)
# ============================================================
class Linear(nn.Module):

    class LinearFunction(torch.autograd.Function):
        @staticmethod
        def forward(ctx, x, W, b):
            ctx.save_for_backward(x, W, b)
            return x @ W + b

        @staticmethod
        def backward(ctx, grad_out):
            x, W, b = ctx.saved_tensors

            grad_x = grad_out @ W.T
            grad_W = x.T @ grad_out
            grad_b = grad_out.sum(0)

            return grad_x, grad_W, grad_b

    def __init__(self, in_features, out_features):
        super().__init__()
        bound = 1 / (in_features ** 0.5)

        self.W = nn.Parameter(torch.empty(in_features, out_features).uniform_(-bound, bound))
        self.b = nn.Parameter(torch.empty(out_features).uniform_(-bound, bound))

    def forward(self, x):
        return Linear.LinearFunction.apply(x, self.W, self.b)



# ============================================================
# Custom MLP
# ============================================================
class My_MLP(nn.Module):
    def __init__(self, input_dim=2, hidden=64, n_classes=3):
        super().__init__()
        self.fc1 = Linear(input_dim, hidden)
        self.fc2 = Linear(hidden, hidden)
        self.fc3 = Linear(hidden, n_classes)

        self.act = ReLU()
        self.loss_fn = CrossEntropyLoss()

    def forward(self, x):
        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        return self.fc3(x)

    def predict(self, x):
        # Automatically convert NumPy → Torch tensor
        if isinstance(x, np.ndarray):
            x = torch.tensor(x, dtype=torch.float32)
    
        with torch.no_grad():
            logits = self.forward(x)
            return torch.argmax(logits, dim=1)




# ============================================================
# Training
# ============================================================
def train_model(X, y, model,optimizer, lr=1e-2, epochs=500):
    

    loss_hist = []
    err_hist  = []

    for epoch in range(epochs):
        logits = model(X)
        loss = model.loss_fn(logits, y)

        # Record training stats
        with torch.no_grad():
            pred = torch.argmax(logits, dim=1)
            mis = (pred != y).float().mean().item()

        loss_hist.append(loss.item())
        err_hist.append(mis)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 100 == 0:
            print(f"epoch {epoch:4d} | loss={loss.item():.4f} | mis={mis:.3f}")

    return loss_hist, err_hist



In [ ]:
# ============================================================
# Train one MLP per dataset + plot all 3 side-by-side
# ============================================================

fig, axes1 = plt.subplots(2, 3, figsize=(12, 6))
fig2, axes2 = plt.subplots(2, 3, figsize=(12, 6))

# Convert numpy → torch float
Xs_t   = torch.tensor(Xs,   dtype=torch.float32)
ys_t   = torch.tensor(ys,   dtype=torch.long)
Xell_t = torch.tensor(Xell, dtype=torch.float32)
yell_t = torch.tensor(yell, dtype=torch.long)
Xsp_t  = torch.tensor(Xsp,  dtype=torch.float32)
ysp_t  = torch.tensor(ysp,  dtype=torch.long)


# ============================================================
# BLOBS
# ============================================================
# --- SGD ---
model_sgd = My_MLP()
optimizer = SGD(model_sgd.parameters())
loss_sgd_blob, err_sgd_blob = train_model(Xs_t, ys_t, model_sgd, optimizer)

with torch.no_grad():
    pred = model_sgd(Xs_t).argmax(dim=1)
mis_sgd = (pred != ys_t).float().mean().item()

plot_decision_boundary(
    Xs_t, ys_t, model_sgd,
    title=f"MLP (SGD, Blobs) err={mis_sgd:.2f}",
    ax=axes1[0,0]
)

# --- Adam ---
model_adam = My_MLP()
optimizer = Adam(model_adam.parameters())
loss_adam_blob, err_adam_blob = train_model(Xs_t, ys_t, model_adam, optimizer)

with torch.no_grad():
    pred = model_adam(Xs_t).argmax(dim=1)
mis_adam = (pred != ys_t).float().mean().item()

plot_decision_boundary(
    Xs_t, ys_t, model_adam,
    title=f"MLP (Adam, Blobs) err={mis_adam:.2f}",
    ax=axes1[1,0]
)

# --- Training Curves ---
plot_all_training_curves(
    axes2[0,0], axes2[1,0],
    loss_sgd_blob, err_sgd_blob,
    loss_adam_blob, err_adam_blob,
    probtype="Blobs"
)


# ============================================================
# ELLIPSOIDS
# ============================================================
# --- SGD ---
model_sgd = My_MLP()
optimizer = SGD(model_sgd.parameters())
loss_sgd_ell, err_sgd_ell = train_model(Xell_t, yell_t, model_sgd, optimizer)

with torch.no_grad():
    pred = model_sgd(Xell_t).argmax(dim=1)
mis_sgd = (pred != yell_t).float().mean().item()

plot_decision_boundary(
    Xell_t, yell_t, model_sgd,
    title=f"MLP (SGD, Ellipsoids) err={mis_sgd:.2f}",
    ax=axes1[0,1]
)

# --- Adam ---
model_adam = My_MLP()
optimizer = Adam(model_adam.parameters())
loss_adam_ell, err_adam_ell = train_model(Xell_t, yell_t, model_adam, optimizer)

with torch.no_grad():
    pred = model_adam(Xell_t).argmax(dim=1)
mis_adam = (pred != yell_t).float().mean().item()

plot_decision_boundary(
    Xell_t, yell_t, model_adam,
    title=f"MLP (Adam, Ellipsoids) err={mis_adam:.2f}",
    ax=axes1[1,1]
)

# --- Training Curves ---
plot_all_training_curves(
    axes2[0,1], axes2[1,1],
    loss_sgd_ell, err_sgd_ell,
    loss_adam_ell, err_adam_ell,
    probtype="Ellipsoids"
)


# ============================================================
# SPIRALS
# ============================================================
# --- SGD ---
model_sgd = My_MLP()
optimizer = SGD(model_sgd.parameters())
loss_sgd_spi, err_sgd_spi = train_model(Xsp_t, ysp_t, model_sgd, optimizer)

with torch.no_grad():
    pred = model_sgd(Xsp_t).argmax(dim=1)
mis_sgd = (pred != ysp_t).float().mean().item()

plot_decision_boundary(
    Xsp_t, ysp_t, model_sgd,
    title=f"MLP (SGD, Spirals) err={mis_sgd:.2f}",
    ax=axes1[0,2]
)

# --- Adam ---
model_adam = My_MLP()
optimizer = Adam(model_adam.parameters())
loss_adam_spi, err_adam_spi = train_model(Xsp_t, ysp_t, model_adam, optimizer)

with torch.no_grad():
    pred = model_adam(Xsp_t).argmax(dim=1)
mis_adam = (pred != ysp_t).float().mean().item()

plot_decision_boundary(
    Xsp_t, ysp_t, model_adam,
    title=f"MLP (Adam, Spirals) err={mis_adam:.2f}",
    ax=axes1[1,2]
)

# --- Training Curves ---
plot_all_training_curves(
    axes2[0,2], axes2[1,2],
    loss_sgd_spi, err_sgd_spi,
    loss_adam_spi, err_adam_spi,
    probtype="Spirals"
)

fig1.tight_layout()
fig2.tight_layout()
fig1.savefig("mlp_decisionboundaries.png", dpi=200, bbox_inches="tight")
fig2.savefig("mlp_trainingcurves.png", dpi=200, bbox_inches="tight")
